# レッスン8: エラーと例外処理 — 壊れないプログラムを書く 🚨

実務のプログラムは「正しい入力が来る」前提では書けません。ファイルが無い、ネットが切れる、ユーザーが変な値を入れる — **異常系にどう備えるかがプロとアマの分かれ目**と言われます。

その前に、まず「エラーメッセージを読める」ようになりましょう。これは**デバッグ(不具合修正)という実務の日常業務**の第一歩です。

---
## 8-1. エラーメッセージの読み方

下のセルはわざとエラーになります。恐れず実行して、出てきた赤い文字を観察してください。

In [ ]:
x = 10
print(y)     # y は定義していないのでエラーになる

```
NameError: name 'y' is not defined
```

読み方のコツ:

1. **最後の行だけ見ればだいたい分かる**。`エラーの種類: 説明` という形式
2. その上の `----> 行番号` が「どの行で起きたか」
3. 長いエラー(Traceback)は「呼び出しの経路」が上から下に並んだもの。**一番下が発生現場**

### 頻出エラー図鑑(この5つで日常の9割)

| エラー | 意味 | よくある原因 |
|--------|------|--------------|
| `SyntaxError` | 文法違反 | コロン忘れ、カッコ閉じ忘れ |
| `NameError` | その名前は知らない | 変数名のタイプミス、定義前に使用 |
| `TypeError` | 型が合わない | 文字列と数値を足した等 |
| `IndexError` | 番号が範囲外 | `T[11]`(要素11個 = 最大は`T[10]`) |
| `ZeroDivisionError` | ゼロ割り | 分母が0になるケースの考慮漏れ |

下の3つのセルで実物を見ておきましょう(全部わざとエラーになります)。

In [ ]:
print("age: " + 20)     # TypeError: 文字列と数値は + でつなげない → str(20) が必要

In [ ]:
T = [0, 1, 2]
print(T[3])              # IndexError: 3要素のインデックスは 0,1,2 まで

In [ ]:
a = 10 / 0               # ZeroDivisionError

---
## 8-2. try / except — エラーが起きても止めない

「エラーになるかもしれない処理」を `try` で囲み、起きたときの対応を `except` に書きます。

In [ ]:
def waru(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("警告: 0では割れません。Noneを返します")
        return None       # None = 「値がない」を表す特別な値

print(waru(10, 2))
print(waru(10, 0))        # プログラムが止まらずに続行できる!
print("処理は最後まで到達しました")

ユーザー入力の検証は実務の定番パターン:

In [ ]:
def to_number(text):
    """文字列を数値に変換。できなければNoneを返す"""
    try:
        return float(text)
    except ValueError:            # 変換できない文字列だとValueErrorが起きる
        return None

for s in ["3.14", "100", "abc", ""]:
    n = to_number(s)
    if n is None:
        print(f"'{s}' は数値に変換できません")
    else:
        print(f"'{s}' → {n}")

---
## 8-3. raise — 自分からエラーを出す

「この値はおかしい、これ以上進むと危険」というとき、**自らエラーを発生させて止める**のも重要な技術です。不正なデータのまま進んで後で静かに壊れるより、早く大きな音で壊れるほうが安全 — 実務では「fail fast(早く失敗せよ)」と呼ばれる原則です。

In [ ]:
def nenrei_touroku(age):
    if age < 0 or age > 150:
        raise ValueError(f"年齢が不正です: {age}")     # エラーを発生させる
    print(f"{age}歳で登録しました")

nenrei_touroku(20)      # OK

try:
    nenrei_touroku(-5)  # ここでValueErrorが飛ぶ
except ValueError as e:  # as e でエラーオブジェクトを受け取れる
    print("登録失敗:", e)

---
## 💼 実務メモ: やってはいけない例外処理

```python
try:
    なにかの処理
except:        # ← 全種類のエラーを黙って握りつぶす
    pass
```

これは**実務で最も嫌われるコード**の一つ。バグが起きても何も表示されず、原因調査が地獄になります。

- exceptには**具体的なエラー種類**を書く(`except ValueError:`)
- 握りつぶさず、**最低限ログ(print)を出す**か、対処できないなら素直にプログラムを止める

---
## ✏️ 練習問題 8-A

次のコードには文法エラーが2つ、実行時エラーが1つ隠れています。**実行してエラーメッセージを読みながら**、3つとも直してください(1つ直すと次のエラーが見えてきます):

```python
data = [10, 20, 30]
for i in range(4)
    print(data[i]
```

エラーメッセージを頼りに直す練習です。「エラーは敵ではなく、場所を教えてくれる味方」を体感しましょう。

In [ ]:
data = [10, 20, 30]
for i in range(4)
    print(data[i]

---
## ✏️ 練習問題 8-B

安全な平均値関数 `heikin(values)` を作ってください:

- リストの平均値(合計 ÷ 個数)を返す
- **空のリスト** `[]` が来たら、エラーで落とさず `None` を返す
- `heikin([1, 2, 3, 4])` → 2.5、`heikin([])` → None を確認

ヒント: 合計は `sum(values)`、個数は `len(values)` で取れます。空リストのとき何が起きるか、まずわざと起こしてみるのがおすすめ。

In [ ]:
# ここにコードを書いてください


---
## ✏️ 練習問題 8-C 【課題コードを頑丈にする】

レッスン3-Cの `dt_max(dx, a)` を強化してください:

- `dx` または `a` が0以下なら `raise ValueError("dxとaは正の値が必要です")` で止める
- 正常な値なら今まで通り `dx**2 / (2*a)` を返す
- 正常ケースと異常ケース(try/exceptで受ける)の両方を試して表示

「入口で値を検証する関数」は、実務コードの最頻出パターンの一つです。

In [ ]:
# ここにコードを書いてください


---
## 🎉 レッスン8はここまで!

**今日覚えたこと:**
1. エラーメッセージは最後の行から読む。頻出5種(Syntax/Name/Type/Index/ZeroDivision)
2. `try / except 種類:` で異常時の対応を書く
3. `raise` で不正な状態を早く・大きく知らせる(fail fast)
4. 「黙って握りつぶす except」は実務のタブー

**次回 → レッスン9: ファイルとモジュール**(データを残す・コードを分ける)

---
### 💡 答え

<details>
<summary>クリックで表示</summary>

```python
# 8-A 修正版
data = [10, 20, 30]
for i in range(3):        # ← コロン追加、range(4)だとIndexErrorなので3に
    print(data[i])        # ← 閉じカッコ追加

# 8-B
def heikin(values):
    if len(values) == 0:
        return None
    return sum(values) / len(values)
# try/except版でもOK:
# def heikin(values):
#     try:
#         return sum(values) / len(values)
#     except ZeroDivisionError:
#         return None

print(heikin([1, 2, 3, 4]))   # 2.5
print(heikin([]))             # None

# 8-C
def dt_max(dx, a):
    if dx <= 0 or a <= 0:
        raise ValueError("dxとaは正の値が必要です")
    return dx ** 2 / (2 * a)

print(dt_max(0.03, 8.264e-7))       # 正常
try:
    dt_max(-0.03, 8.264e-7)         # 異常
except ValueError as e:
    print("エラー捕捉:", e)
```
</details>